In [288]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np
import pandas as pd
from scipy.interpolate import griddata

In [289]:
df = pd.read_csv('results.csv')
# df = pd.read_csv('AUBC_test.csv')

df.columns.values[0] = 'iterations'

print(df.shape)
df.head()

(754, 4)


,iterations,L2_value,drop_value,loss
0,0,0.000162,0.348326,0.636713
1,1,0.001153,0.137565,0.640184
2,2,0.001867,0.408430,0.645374
3,3,0.000098,0.002429,0.637338
4,4,0.000004,0.136765,0.634278


In [290]:
# Calculate the interval between each instance
df['interval'] = df['iterations'].diff().fillna(0)

# Calculate the trapezoidal area
df.loc[:, 'trapezoidal_area'] = 0.5 * (df['loss'] + df['loss'].shift(1, fill_value=0)) * df['interval']

df['cumulative_area'] = df['trapezoidal_area'].cumsum()

df['AUBC'] = df['cumulative_area'] / (df['cumulative_area'].max() * df.shape[0])

df['AUBC_totals'] = df['AUBC'].cumsum()

df

,iterations,L2_value,drop_value,loss,interval,trapezoidal_area,cumulative_area,AUBC,AUBC_totals
0,0,0.000162,0.348326,0.636713,0.0,0.000000,0.000000,0.000000,0.000000
1,1,0.001153,0.137565,0.640184,1.0,0.638448,0.638448,0.000002,0.000002
2,2,0.001867,0.408430,0.645374,1.0,0.642779,1.281227,0.000004,0.000005
3,3,0.000098,0.002429,0.637338,1.0,0.641356,1.922583,0.000005,0.000011
4,4,0.000004,0.136765,0.634278,1.0,0.635808,2.558391,0.000007,0.000018
...,...,...,...,...,...,...,...,...,...
749,749,0.000002,0.400326,0.629485,1.0,0.631906,478.996736,0.001319,0.496022
750,750,0.000014,0.461910,0.633757,1.0,0.631621,479.628357,0.001321,0.497343
751,751,0.000067,0.214278,0.630330,1.0,0.632044,480.260401,0.001323,0.498666
752,752,0.000080,0.086990,0.632378,1.0,0.631354,480.891755,0.001325,0.499990


In [291]:
def plot_3d_scatter_surface(df, x_variable, y_variables, output_variable, surface=True, highlight=False):
    name = ['L2', 'Dropout']
    
    for i, y_var in enumerate(y_variables):
        print(f'{name[i]}: Min value = {df[y_var].min()}, Max value = {df[y_var].max()}, Range = {df[y_var].max() - df[y_var].min()}')
        print(f'Min Loss = {df[output_variable].min()}')
        # Find the index of the lowest output variable value if highlight is True
        lowest_index = df[output_variable].idxmin() if highlight else None
        
        # Create a scatter plot
        scatter = go.Scatter3d(
            x=df[x_variable],
            y=df[y_var],
            z=df[output_variable],
            mode='markers',
            marker=dict(
                size=[15 if i == lowest_index else 5 for i in range(len(df))],  # Highlighted point is bigger,
                color=['red' if i == lowest_index else 'grey' for i in range(len(df))] if highlight else 'grey',  # Highlight the lowest value if highlight is True
                colorscale='Viridis',  # Choose a color scale
                opacity=0.8,
            ),
            name=output_variable
        )

        # Create surface data
        x_data = df[x_variable]
        y_data = df[y_var]
        z_data = df[output_variable]

        x_range = np.linspace(min(x_data), max(x_data), 100)
        y_range = np.linspace(min(y_data), max(y_data), 100)
        X, Y = np.meshgrid(x_range, y_range)
        Z = griddata((x_data, y_data), z_data, (X, Y), method='cubic')

        if surface:
            # Create a 3D surface
            surface = go.Surface(
                x=X,
                y=Y,
                z=Z,
                colorscale='Jet',  # Choose a color scale
                opacity=0.6
            )

            # Create the figure
            fig = go.Figure(data=[scatter, surface])

        else:
            fig = go.Figure(data=[scatter])

        # Update layout
        fig.update_layout(
            scene=dict(
                xaxis_title=x_variable,
                yaxis_title=y_var,
                zaxis_title=output_variable
            ),
            width=800,  # adjust width
            height=600,  # adjust height
            title=f'Mean Loss for different values of {name[i]} regularization'
        )

        # Show the plot
        fig.show()

        # Save the plot to an HTML file
        file_name = f'3D_plot_{output_variable}_vs_{y_var}_plot.html'
        pio.write_html(fig, file_name)

In [292]:
plot_3d_scatter_surface(df, 'iterations', ['L2_value', 'drop_value'], 'loss', surface=False, highlight=True)

L2: Min value = 1.05e-06, Max value = 0.004160265, Range = 0.004159215
Min Loss = 0.619165382


Dropout: Min value = 0.001647456, Max value = 0.499534249, Range = 0.497886793
Min Loss = 0.619165382


In [293]:
def plot_2d_scatter(df, x_column, output_column, highlight=False):

    # Find the index of the lowest output variable value if highlight is True
    lowest_index = df[output_column].idxmin() if highlight else None
    
    # Create a scatter plot
    scatter = go.Scatter(
        x=df[x_column],
        y=df[output_column],
        mode='lines+markers',
    
        marker=dict(
            size=[15 if i == lowest_index else 5 for i in range(len(df))],  # Highlighted point is bigger
            color=['red' if i == lowest_index else 'blue' for i in range(len(df))] if highlight else 'blue',  # Highlight the lowest value if highlight is True
            opacity=0.8,
        ),
        name=output_column
    )

    # Create the figure
    fig = go.Figure(data=[scatter])

    # Update layout
    fig.update_layout(
        xaxis_title=x_column,
        yaxis_title=output_column,
        # yaxis=dict(tickformat='e'),
        width=800,  # adjust width
        height=600,  # adjust height
        title=f'{output_column} as a function of {x_column}'
    )

    # Show the plot
    fig.show()

    # Save the plot to an HTML file
    file_name = f'{output_column}_vs_{x_column}_plot.html'
    pio.write_html(fig, file_name)

In [294]:
plot_2d_scatter(df, 'iterations', 'loss', highlight=True)

In [295]:
plot_2d_scatter(df, 'iterations', 'L2_value', highlight=True)


In [296]:
plot_2d_scatter(df, 'iterations', 'drop_value', highlight=True)

In [297]:
print(f"Area under the curve (AUC): {df['AUBC_totals'].iloc[-1]:.3f}")

plot_2d_scatter(df, 'iterations', 'AUBC', highlight=False)

Area under the curve (AUC): 0.501
